# 09 - Full Walk-Forward Validation & Parameter Grid Search

**Author:** Sacha Huberty

**Purpose:** Formalize the S10 validation discipline the project has been
building toward: grid search two cheap-to-tune parameter families
in-sample only (mean-reversion lookback/entry_z via a no-model-fitting
event study, and turnover cap/no-trade band via a memoized
Black-Litterman strategy), select on cross-fold *stability*, not peak
performance, freeze the winning values into `config.yaml`, and then run
that frozen config through the out-of-sample walk-forward exactly once.
HMM and autoencoder hyperparameters are intentionally NOT grid-searched
here (prohibitive refit cost at this scale) and stay at their existing,
reasoned stage 3/4 defaults.

**Last updated:** 2026-07-26

**Note on the grid-search methodology used here:** the literal
"re-fit every model per fold" walk-forward described in
PROJECT_STRUCTURE.md 7 would need a fresh HMM + autoencoder refit per
parameter combination per fold, which is computationally infeasible in
this environment. Instead: (1) mean-reversion tuning uses
`meanreversion`'s pure vectorized math (no model fitting) so all 9
combos across all 12 IS years cost under a second; (2) turnover tuning
memoizes ONE full run of the (already-fitted-as-usual) Black-Litterman
strategy over the last 4 in-sample years, then replays cheap
accounting-only re-runs for each of the 9 rebalance-parameter combos.
This is faithful to the stage's spirit (IS-only tuning selected for
cross-fold stability, frozen config validated OOS exactly once) while
staying tractable.

## Setup

In [ ]:
# Must run before numpy/scipy/tensorflow are imported: on Windows, this
# backtest makes hundreds of tiny (~22-asset) SLSQP/HMM/covariance calls
# in a tight loop, and letting OpenBLAS/MKL and TensorFlow's own thread
# pools both fight over all CPU cores causes intermittent multi-minute
# stalls on individual weeks (confirmed empirically: identical code was
# ~9x slower and had several 30+ minute outlier calls with default
# threading vs. forced single-threaded BLAS -- stage 9). Each week's
# problem is small enough that single-threaded is strictly faster here.
import os

for _var in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS",
):
    os.environ[_var] = "1"

import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import (
    allocation, backtest, data, meanreversion, metrics, regimes, strategy,
    universe,
)

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["grid_search"]

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)

is_end = pd.Timestamp(cfg["general"]["is_end_date"])
oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
lookback = cfg["optimization"]["lookback_days"]
cov_method = cfg["optimization"]["covariance"]

is_prices = prices.loc[:is_end]
returns.tail()

## Analysis / signal logic

### A. IS-only grid search: mean-reversion hyperparameters (event study)

No model fitting involved (`meanreversion.zscore` is pure rolling-window
math), so this is cheap enough to run the full mean-reversion grid
(`grid_search.meanreversion`) across every annual in-sample fold
(2010-2021). For each `(lookback_days, entry_z)` combo, in each annual
fold: was a >|entry_z| deviation from trend actually followed by
reversion toward the mean over the next 20 trading days? The z-score
and forward return are computed on the FULL in-sample price history
(so the rolling window has real data before the fold starts), but only
events dated inside the fold are counted toward that fold's hit rate --
this isolates a genuine per-year read rather than a growing-window
cumulative one.

In [ ]:
def fold_hit_rate(prices_full, combo_cfg, fold_start, fold_end, horizon_days=20):
    mcfg = combo_cfg["meanreversion"]
    window = mcfg["lookback_days"]
    log_price = np.log(prices_full)
    z = meanreversion.zscore(log_price, window)
    future_change = log_price.shift(-horizon_days) - log_price
    reverted = np.sign(future_change) == -np.sign(z)
    extreme = z.abs() > mcfg["entry_z"]

    in_fold = (z.index >= fold_start) & (z.index <= fold_end)
    extreme_fold = extreme.loc[in_fold]
    valid_fold = future_change.loc[in_fold].notna()
    reverted_fold = reverted.loc[in_fold]

    eligible = extreme_fold & valid_fold
    n = int(eligible.to_numpy().sum())
    if n == 0:
        return np.nan, 0
    hits = int((eligible & reverted_fold).to_numpy().sum())
    return hits / n, n


fold_periods = is_prices.index.to_period("Y").unique()
penalty = cfg["grid_search"]["stability_penalty"]
min_events_per_fold = 5


def evaluate_meanreversion(params):
    combo_cfg = copy.deepcopy(cfg)
    combo_cfg["meanreversion"]["lookback_days"] = params["lookback_days"]
    combo_cfg["meanreversion"]["entry_z"] = params["entry_z"]

    hit_rates, total_events = [], 0
    for period in fold_periods:
        hr, n = fold_hit_rate(
            is_prices, combo_cfg, period.start_time, period.end_time
        )
        total_events += n
        if n >= min_events_per_fold:
            hit_rates.append(hr)

    hit_rates = pd.Series(hit_rates, dtype=float)
    mean_hr = hit_rates.mean() if len(hit_rates) else np.nan
    std_hr = hit_rates.std(ddof=1) if len(hit_rates) > 1 else 0.0
    return {
        "mean_hit_rate": mean_hr,
        "hit_rate_std": std_hr,
        "stability_score": mean_hr - penalty * std_hr,
        "total_events": total_events,
        "n_folds_with_events": len(hit_rates),
    }


meanrev_grid = backtest.grid_search(
    cfg["grid_search"]["meanreversion"], evaluate_meanreversion
)
meanrev_grid.sort_values("stability_score", ascending=False)

In [ ]:
# Require a reasonable sample size before trusting a combo's hit rate.
qualified = meanrev_grid[meanrev_grid["total_events"] >= 30]
best_meanrev = qualified.sort_values("stability_score", ascending=False).iloc[0]
best_meanrev

### B. IS-only grid search: turnover parameters (memoized BL strategy)

The turnover cap and no-trade band are rebalance-time-only parameters:
they never change what the strategy WANTS to hold, only how quickly it
gets there. That means the (expensive) weekly target-weight sequence
can be computed once via `backtest.memoize_strategy` and reused for
every `(no_trade_band, max_weekly_turnover)` combo -- each grid point
then only re-runs `backtest.run`'s cheap accounting loop.

**Scoping note:** to keep this tractable, this grid search uses only
the last 4 in-sample years (still strictly before `oos_start_date`,
so still IS-only) rather than the full 2010-2021 span used in part A --
running the full BL pipeline (HMM decode + views + SLSQP) week-by-week
over 12 years just to tune 2 rebalance parameters is not worth the
cost. The anomaly override is also left off here (it doesn't interact
with turnover-cap mechanics) to save the autoencoder's fit cost; it is
restored for the final frozen OOS run in part D, matching stage 8's
"current best" pipeline.

In [ ]:
tuned_cfg = copy.deepcopy(cfg)
tuned_cfg["meanreversion"]["lookback_days"] = int(best_meanrev["lookback_days"])
tuned_cfg["meanreversion"]["entry_z"] = float(best_meanrev["entry_z"])

turnover_is_start = is_end - pd.DateOffset(years=4)
buffer_start_pos = max(0, returns.index.searchsorted(turnover_is_start) - lookback)
turnover_is_returns = returns.iloc[buffer_start_pos:returns.index.searchsorted(is_end) + 1]

bl_fn_for_turnover = strategy.black_litterman_strategy(
    class_bucket, tuned_cfg, posture_cfg
)
memoized_bl_fn = backtest.memoize_strategy(bl_fn_for_turnover)

is_validation_fold = cfg["grid_search"]["is_validation_fold"]


def evaluate_turnover(params):
    combo_cfg = copy.deepcopy(tuned_cfg)
    combo_cfg["rebalance"]["no_trade_band"] = params["no_trade_band"]
    combo_cfg["rebalance"]["max_weekly_turnover"] = params["max_weekly_turnover"]

    result = backtest.run(memoized_bl_fn, turnover_is_returns, combo_cfg)
    is_returns_only = result.daily_returns.loc[turnover_is_start:]
    wf = backtest.walk_forward(is_returns_only, combo_cfg, fold=is_validation_fold)
    return {
        **wf.summary,
        "avg_weekly_turnover": result.metrics["avg_weekly_turnover"],
        "total_cost_drag": result.metrics["total_cost_drag"],
    }


turnover_grid = backtest.grid_search(cfg["grid_search"]["turnover"], evaluate_turnover)
turnover_grid.sort_values("stability_score", ascending=False)

In [ ]:
best_turnover = turnover_grid.sort_values("stability_score", ascending=False).iloc[0]
best_turnover

### C. Freeze the config

HMM (`regimes.hmm`) and autoencoder (`anomaly`) hyperparameters are
deliberately left untouched at their stage 3/4 defaults: exhaustively
grid-searching them would mean re-fitting an HMM and an LSTM
autoencoder per parameter combination per fold, which is not
tractable here. They remain reasoned choices, not tuned ones -- this
is stated plainly rather than dressed up as a completed grid search.

In [ ]:
frozen_cfg = copy.deepcopy(cfg)
frozen_cfg["meanreversion"]["lookback_days"] = int(best_meanrev["lookback_days"])
frozen_cfg["meanreversion"]["entry_z"] = float(best_meanrev["entry_z"])
frozen_cfg["rebalance"]["no_trade_band"] = float(best_turnover["no_trade_band"])
frozen_cfg["rebalance"]["max_weekly_turnover"] = float(best_turnover["max_weekly_turnover"])

print("Frozen meanreversion:", frozen_cfg["meanreversion"]["lookback_days"], frozen_cfg["meanreversion"]["entry_z"])
print("Frozen rebalance:", frozen_cfg["rebalance"]["no_trade_band"], frozen_cfg["rebalance"]["max_weekly_turnover"])

### D. Frozen config -> OOS walk-forward, run exactly once

This is the canonical result. The anomaly override is restored here
(matching stage 8's "current best" pipeline: BL + anomaly override).
Same buffered-lookback, reduced-anomaly-epoch scope trade as stages
6-8 (a compute-budget concession applied identically to every strategy
compared below, not chosen in response to OOS performance). After this
cell runs, `frozen_cfg`'s parameters are not revisited.

In [ ]:
buffer_start_pos = max(0, returns.index.searchsorted(oos_start) - lookback)
backtest_returns = returns.iloc[buffer_start_pos:]

oos_cfg = copy.deepcopy(frozen_cfg)
oos_cfg["anomaly"]["epochs"] = 10
oos_cfg["anomaly"]["patience"] = 3
oos_cfg["anomaly"]["refit_frequency_days"] = 126

bl_fn = strategy.black_litterman_strategy(class_bucket, oos_cfg, posture_cfg)
bl_anomaly_fn = strategy.with_anomaly_override(bl_fn, oos_cfg)

frozen_result = backtest.run(bl_anomaly_fn, backtest_returns, oos_cfg)

In [ ]:
snapshot_window = returns.loc[:is_end].tail(lookback)
mu_is = allocation.mean_returns(snapshot_window)
cov_is = allocation.covariance_matrix(snapshot_window, method=cov_method)
w_max_sharpe_static = allocation.max_sharpe(mu_is, cov_is, cfg)


def make_classical_strategy(method):
    def strategy_fn(as_of, window):
        w = window.tail(lookback)
        cov_t = allocation.covariance_matrix(w, method=cov_method)
        if method == "gmv":
            return allocation.gmv(cov_t, cfg)
        if method == "risk_parity":
            return allocation.risk_parity(cov_t, cfg)
        if method == "hrp":
            return allocation.hrp(w, cfg)
        raise ValueError(method)
    return strategy_fn


def permanent_strategy(as_of, window):
    return allocation.permanent(class_bucket)


def sixty_forty_strategy(as_of, window):
    return allocation.sixty_forty(class_bucket)


def max_sharpe_static_strategy(as_of, window):
    return w_max_sharpe_static


benchmark_fns = {
    "permanent": permanent_strategy,
    "sixty_forty": sixty_forty_strategy,
    "max_sharpe_static": max_sharpe_static_strategy,
    "gmv": make_classical_strategy("gmv"),
    "risk_parity": make_classical_strategy("risk_parity"),
    "hrp": make_classical_strategy("hrp"),
}
benchmark_results = {
    name: backtest.run(fn, backtest_returns, cfg)
    for name, fn in benchmark_fns.items()
}
all_results = {"black_litterman_frozen": frozen_result, **benchmark_results}

## Results

In [ ]:
def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "sortino": metrics.sortino(r),
        "calmar": metrics.calmar(r),
        "max_drawdown": metrics.max_drawdown(r),
        "hit_rate": metrics.hit_rate(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


comparison_table = pd.DataFrame(
    {name: oos_metrics(res) for name, res in all_results.items()}
).T
comparison_table.sort_values("sharpe", ascending=False)

### Per-fold OOS walk-forward (the canonical, stability-checked view)

Quarterly folds (`backtest.walk_forward_fold`) on the frozen strategy's
OOS daily returns -- the actual S10 "average over folds" report, not
just one aggregate OOS number.

In [ ]:
frozen_oos_returns = frozen_result.daily_returns.loc[oos_start:]
frozen_wf = backtest.walk_forward(frozen_oos_returns, oos_cfg)
frozen_wf.fold_metrics

In [ ]:
frozen_wf.summary

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in all_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    lw = 2 if name == "black_litterman_frozen" else 1
    oos_curve.plot(label=name, linewidth=lw)
plt.title("OOS equity curves: frozen Black-Litterman vs. benchmarks")
plt.ylabel("Growth of $1, rebased to OOS start")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(11, 4))
for name, res in all_results.items():
    dd = metrics.drawdown_series(res.daily_returns.loc[oos_start:])
    lw = 2 if name == "black_litterman_frozen" else 1
    dd.plot(label=name, linewidth=lw)
plt.title("OOS drawdowns: frozen Black-Litterman vs. benchmarks")
plt.ylabel("Drawdown")
plt.legend()
plt.show()

In [ ]:
frozen_result.weights.loc[oos_start:].plot.area(
    figsize=(11, 5), title="Frozen Black-Litterman: OOS weight evolution"
)
plt.ylabel("Weight")
plt.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

### Regime-colored performance (visualization context only)

A single full-sample HMM fit/decode over the whole return history,
used only to shade the background by posture for visual context. This
is NOT the walk-forward, point-in-time posture the strategy actually
used week to week (that is recomputed on a rolling basis inside
`black_litterman_strategy`) -- it's a cheaper, full-sample view purely
to help the eye connect performance with regime, and should not be
read as a performance number.

In [ ]:
market_ticker = cfg["regimes"]["market_ticker"]
full_regime = regimes.market_regime(returns[market_ticker], cfg, posture_cfg)
posture_series = full_regime["posture_series"].reindex(frozen_oos_returns.index).ffill()

posture_colors = {"risk_on": "#c8e6c9", "risk_off": "#ffcdd2", "neutral": "#e0e0e0"}
fig, ax = plt.subplots(figsize=(11, 5))
equity = (1.0 + frozen_oos_returns).cumprod()
prev_date = posture_series.index[0]
prev_posture = posture_series.iloc[0]
for date, posture in posture_series.items():
    if posture != prev_posture:
        ax.axvspan(prev_date, date, color=posture_colors.get(prev_posture, "#ffffff"), alpha=0.5)
        prev_date, prev_posture = date, posture
ax.axvspan(prev_date, posture_series.index[-1], color=posture_colors.get(prev_posture, "#ffffff"), alpha=0.5)
equity.plot(ax=ax, color="black", linewidth=1.5, label="black_litterman_frozen")
ax.set_title("Frozen Black-Litterman OOS equity, shaded by full-sample HMM posture")
ax.legend()
plt.tight_layout()
plt.show()

### Turnover report

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
frozen_result.turnover.loc[oos_start:].plot(ax=ax, label="weekly turnover")
ax.axhline(oos_cfg["rebalance"]["max_weekly_turnover"], color="red", linestyle="--", label="turnover cap")
ax.axhline(oos_cfg["rebalance"]["no_trade_band"], color="gray", linestyle=":", label="no-trade band")
ax.set_title("Frozen Black-Litterman: OOS weekly turnover vs. cap and no-trade band")
ax.legend()
plt.tight_layout()
plt.show()

## Notes / next steps

In [ ]:
print(f"Winning meanreversion combo: lookback_days={frozen_cfg['meanreversion']['lookback_days']}, entry_z={frozen_cfg['meanreversion']['entry_z']}")
print(f"Winning turnover combo: no_trade_band={frozen_cfg['rebalance']['no_trade_band']}, max_weekly_turnover={frozen_cfg['rebalance']['max_weekly_turnover']}")
print(comparison_table.sort_values('sharpe', ascending=False))

**Honest findings:**

- **The grid search did not improve OOS performance over stage 8's
  manually-chosen defaults, and if anything is marginally behind.**
  The IS-selected combo (mean-reversion `lookback_days=20, entry_z=2.0`;
  turnover `no_trade_band=0.01, max_weekly_turnover=0.01`) produced a
  frozen Black-Litterman OOS Sharpe of **0.8564** over 2022-01 to
  2026-07, versus stage 8's untuned 0.895 over a similar (shorter)
  OOS window. This is exactly what the S10 discipline predicts can
  happen: selecting IS parameters for cross-fold *stability* rather
  than peak IS performance is not a promise of better OOS performance,
  and the honest-reporting rule means saying so plainly rather than
  re-tuning after the fact.
- `permanent` again posts the best OOS Sharpe of the whole comparison
  (1.0983), same as stage 8's finding. The frozen Black-Litterman
  strategy's max drawdown (-10.4%) is close to but not quite better
  than permanent's (-12.6%), and is not the best in the group either
  (HRP's -9.2% is the smallest). BL's annualized vol (5.3%) remains
  the second-lowest after HRP, consistent with the utility gate
  sometimes preferring the defensive GMV/Risk-Parity book.
- **The turnover grid search had limited power to discriminate.**
  All 9 combos' stability scores were negative and clustered close
  together (-0.607 to -0.783); the top 3 combos (all using the
  tightest 1% turnover cap) differed by less than 0.003 in stability
  score, indistinguishable given the underlying noise. Only 4 annual
  IS folds were available for this grid (the scoped-down last-4-years
  window used to keep the memoized run affordable), which is a small
  sample for a std-penalized stability score. The winning combo does
  cleanly minimize average turnover (1.13%) and cost drag (0.41%)
  versus the wider bands, which is a real, if modest, effect.
- **Quarterly OOS fold Sharpe is highly volatile and its mean should
  not be read as "the" Sharpe.** The per-fold table shows sharpe_mean
  1.42 vs. sharpe_std 2.21 (stability_score -0.79): 2022 was sharply
  negative (Q1-Q3 Sharpe -0.78, -2.46, -2.28 amid the bond+equity
  selloff) before a strong recovery, and several quarters since show
  Sharpe above 4-5, which mostly reflects how noisy an annualized
  Sharpe estimated from ~62 days of data is, not sustained skill. The
  properly-computed full-period OOS Sharpe (0.86, from the whole
  contiguous return series) is the reliable headline number; the
  fold-by-fold average is for reading consistency and dispersion, not
  as a better point estimate.
- **A real engineering finding from this stage, independent of the
  strategy's numbers:** two of the four IS validation folds tested
  during grid search (2018-03 and 2018-04, under the eventual
  winning `lookback_days=20`) took 30+ minutes each in a single-
  threaded-unaware run, driven by OpenBLAS/MKL and TensorFlow's
  thread pools fighting over CPU cores across hundreds of tiny
  (~22-asset) SLSQP/HMM calls in a tight Windows loop -- not a bug in
  the strategy logic. Forcing single-threaded BLAS/OMP (top of the
  Setup cell) fixed it and made the whole notebook roughly 9x faster
  end to end, since each week's optimization problem is far too small
  to benefit from multi-threading anyway.

